# CASTalign Testground

Streamlined alignment workflow for 2D histology slices → 3D in-vivo volume.

## Workflow Overview
1. **Load graph** with in-vivo reference + subslices
2. **Select slice** via dropdown widget
3. **Align** using one of three modes:
   - Mode A: Padded, 3D as base (point-based, RAM-heavy)
   - Mode B: No padding, 3D as base (parametric only, low RAM)
   - Mode C: No padding, 2D as base / 3D movable (point-based, low RAM)
4. **Refine** with Triangulation2D for edge warping
5. **Visualize** all aligned slices with GraphViewer

---

## 1. Imports & Setup

In [ ]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================
%reset -f

import sys
import platform
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# =============================================================================
# PLATFORM DETECTION & PATH SETUP
# =============================================================================
IS_MAC = platform.system() == "Darwin"
IS_LINUX = platform.system() == "Linux"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # linestuffup_dtc/

# Add project utilities to path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# =============================================================================
# CASTALIGN LIBRARY PATH (platform-specific)
# =============================================================================
if IS_MAC:
    # Mac: Local castalign installation
    CASTALIGN_ROOT = Path("/Users/dtc32/Library/CloudStorage/OneDrive-DukeUniversity/Documents/Medical School/PhD/GS 1/Li Lab/programs/castalign")
else:
    # Linux server
    CASTALIGN_ROOT = Path("/home/dtc/lab/programs/castalign")

if str(CASTALIGN_ROOT) not in sys.path:
    sys.path.insert(0, str(CASTALIGN_ROOT))

print(f"Platform: {'macOS' if IS_MAC else 'Linux'}")
print(f"CASTalign: {CASTALIGN_ROOT}")

# =============================================================================
# CASTALIGN IMPORTS
# =============================================================================
try:
    import castalign as ca
    import castalign.gui as ca_gui
    from castalign.gui import GraphViewer
    from castalign import ndarray_shifted
    import castalign.utils as ca_utils
    print("Imported castalign")
except ImportError:
    # Fallback to linestuffup if castalign not available
    import linestuffup as ca
    import linestuffup.gui as ca_gui
    from linestuffup import ndarray_shifted
    import linestuffup.utils as ca_utils
    try:
        from linestuffup.gui import GraphViewer
    except ImportError:
        GraphViewer = None
        print("Warning: GraphViewer not available in this version")
    print("Imported linestuffup (castalign fallback)")

# =============================================================================
# PROJECT UTILITIES
# =============================================================================
from utilities.graph_viz_fix import patch_castalign_visualise
patch_castalign_visualise()

# =============================================================================
# JUPYTER MAGIC
# =============================================================================
%load_ext autoreload
%autoreload 2

# Disable label auto-detection
ca_utils.image_is_label = lambda img: False

print()
print("="*60)
print("CASTALIGN TESTGROUND")
print("="*60)
print("Imports complete")
print("GraphViz patched (SVG output)")
print("Auto-reload enabled")
print("="*60)

## 2. Path Configuration

In [ ]:
# =============================================================================
# PATH CONFIGURATION (platform-aware)
# =============================================================================
# 
# WARNING: SQLite over Samba (Mac accessing server files)
# --------------------------------------------------------
# The graph .db files are SQLite databases. Writing SQLite over network mounts
# (Samba/SMB) can sometimes cause issues:
#   - "database is locked" errors
#   - Corruption errors on save
#   - Hanging during graph.save()
# 
# If you encounter these issues, consider:
#   1. Copy graph locally, align, copy back
#   2. Run alignment directly on the server (SSH + X11 forwarding)
#
# Initial file loading (200MB in vivo stack) may take 30-60 seconds over network,
# but once in RAM, napari will be smooth.
# =============================================================================

if IS_MAC:
    # ==========================================================================
    # MAC PATHS (accessing server data via Samba mount)
    # ==========================================================================
    # Server data mounted via Samba
    LAB_ROOT = Path("/Volumes/home/lab")
    
    # TODO: Adjust these Mac-local paths as needed
    # PROJECT_ROOT for utilities is set in imports cell based on notebook location
    
else:
    # ==========================================================================
    # LINUX SERVER PATHS
    # ==========================================================================
    LAB_ROOT = Path("/home/dtc/lab")

# =============================================================================
# DATA PATHS (relative to LAB_ROOT - same structure on both platforms)
# =============================================================================
OUTPUT_ROOT = LAB_ROOT / "output"

# In-vivo reference
INVIVO_STACK_PATH = OUTPUT_ROOT / "in_vivo_flip_corrected" / "JH302_1x_ch2_flipped.tiff"

# Subslice directory (anisotropic cell mask overlays)
ANISO_SUBSLICE_DIR = OUTPUT_ROOT / "mScarlet_cellmask_subslice" / "threshold_0.30_cellmask_0.50_anisotropic"

# Graph paths (read AND write to server)
GRAPH_DIR = OUTPUT_ROOT / "linestuffup_output"
GRAPH_PATH = GRAPH_DIR / "castalign_test.db"

# =============================================================================
# PATH VERIFICATION
# =============================================================================
print(f"LAB_ROOT: {LAB_ROOT}")
print()
print("Path verification:")
print(f"  In-vivo stack: {'OK' if INVIVO_STACK_PATH.exists() else 'NOT FOUND'} {INVIVO_STACK_PATH.name}")
print(f"  Subslice dir:  {'OK' if ANISO_SUBSLICE_DIR.exists() else 'NOT FOUND'} {ANISO_SUBSLICE_DIR.name}")
print(f"  Graph file:    {'OK' if GRAPH_PATH.exists() else 'WILL BE CREATED'} {GRAPH_PATH.name}")

if ANISO_SUBSLICE_DIR.exists():
    n_slices = len(list(ANISO_SUBSLICE_DIR.glob("*.tif")))
    print(f"\n  Found {n_slices} subslice files")

if IS_MAC:
    print()
    print("NOTE: Running on Mac - data accessed via Samba mount")
    print("      Initial load may take 30-60 sec over network")

## 3. Load Graph

In [ ]:
# =============================================================================
# LOAD GRAPH
# =============================================================================

if not GRAPH_PATH.exists():
    raise FileNotFoundError(f"Graph not found: {GRAPH_PATH}\nRun subslice_graph_builder.py to build it first.")

g = ca.Graph.load(str(GRAPH_PATH))

# Get slice info
all_nodes = list(g.nodes)
invivo_node = "invivo_ref"

# Find slice nodes (all nodes except invivo_ref)
# Node naming: slice{N}_subslice_mScarlet_cellmask
slice_nodes = sorted([n for n in all_nodes if n != invivo_node])

# Check which slices are already aligned
aligned_slices = []
unaligned_slices = []
for s in slice_nodes:
    if s in g.edges and invivo_node in g.edges[s]:
        aligned_slices.append(s)
    else:
        unaligned_slices.append(s)

print("="*60)
print("GRAPH LOADED")
print("="*60)
print(f"Graph: {g.name}")
print(f"Total nodes: {len(all_nodes)}")
print(f"Subslices: {len(slice_nodes)}")
print(f"  Aligned: {len(aligned_slices)}")
print(f"  Unaligned: {len(unaligned_slices)}")
print("="*60)

## 4. Slice Selector

Use the dropdown to select which slice to align. Green = aligned, Red = unaligned.

In [ ]:
# =============================================================================
# SLICE SELECTOR WIDGET
# =============================================================================
import re

def extract_slice_number(node_name):
    """Extract slice number from node name like 'slice10_subslice_mScarlet_cellmask'."""
    match = re.match(r'slice(\d+)', node_name)
    return int(match.group(1)) if match else 0

# Sort slice nodes by number
slice_nodes = sorted(slice_nodes, key=extract_slice_number)

# Create options with alignment status
def get_slice_options():
    """Generate dropdown options with alignment status."""
    options = []
    for s in slice_nodes:
        is_aligned = s in aligned_slices
        status = "OK" if is_aligned else "o"
        num = extract_slice_number(s)
        options.append((f"{status} Slice {num}", s))
    return options

# Create widget
slice_dropdown = widgets.Dropdown(
    options=get_slice_options(),
    description="Select:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="250px")
)

# Info display
info_output = widgets.Output()

def show_slice_info(node):
    """Display info for a slice node."""
    with info_output:
        clear_output(wait=True)
        img = g.get_image(node)
        is_aligned = node in aligned_slices
        
        print(f"Selected: {node}")
        print(f"Shape: {img.shape}")
        print(f"Status: {'Aligned' if is_aligned else 'Not aligned'}")
        
        if is_aligned:
            t = g.get_transform(node, invivo_node)
            print(f"Transform: {type(t).__name__}")

def on_slice_change(change):
    """Update info when slice selection changes."""
    show_slice_info(change["new"])

# Attach observer FIRST
slice_dropdown.observe(on_slice_change, names="value")

# Display widget
print("Select a slice to align:")
print("  OK = already aligned")
print("  o  = not yet aligned")
print()
display(widgets.HBox([slice_dropdown, info_output]))

# Show initial info (observer won't fire since value hasn't changed)
show_slice_info(slice_dropdown.value)

def get_selected_slice():
    """Get currently selected slice name."""
    return slice_dropdown.value

## 5. Alignment Utilities

Helper functions used by all alignment modes.

In [ ]:
# =============================================================================
# ALIGNMENT UTILITIES
# =============================================================================

def get_initial_transform_3d_base():
    """
    Initial transform for 3D-as-base mode.
    
    Applies:
    - xrotate=90: Maps ex-vivo Y -> in-vivo Z (dorsal stays on top)
    - FlipX: Reverses left-right to match in-vivo orientation
    """
    rotation = ca.TranslateRotateFixed(xrotate=90)
    flip = ca.FlipFixed(x=True)
    return rotation + flip


def get_initial_transform_3d_movable():
    """
    Initial transform for 3D-as-movable mode.
    
    TODO: This needs adjustment - the orientation flip may differ
    when the 3D volume is movable. Return to this after testing.
    """
    # Placeholder - will need adjustment based on testing
    # The transform direction is inverted, so flips may differ
    return ca.Identity()


def save_alignment(slice_name, transform):
    """
    Save alignment to graph and disk.
    
    Parameters
    ----------
    slice_name : str
        Name of the slice node
    transform : Transform
        The fitted transform (slice -> invivo_ref direction)
    """
    global aligned_slices, unaligned_slices
    
    # Add/update edge
    try:
        g.add_edge(slice_name, invivo_node, transform, update=True)
    except:
        g.add_edge(slice_name, invivo_node, transform)
    
    # Save to disk
    g.save()
    
    # Update tracking lists
    if slice_name not in aligned_slices:
        aligned_slices.append(slice_name)
    if slice_name in unaligned_slices:
        unaligned_slices.remove(slice_name)
    
    # Update dropdown options
    slice_dropdown.options = get_slice_options()
    
    print(f"\nSaved: {slice_name} -> {invivo_node}")
    print(f"  Graph saved to: {GRAPH_PATH.name}")


def get_previous_transform(slice_name):
    """
    Get transform from a nearby aligned slice for rotation carry-forward.
    
    Returns the transform from the closest lower-numbered aligned slice,
    or None if no previous slices are aligned.
    """
    # Extract slice number using the helper function
    current_num = extract_slice_number(slice_name)
    if current_num == 0:
        return None
    
    # Find closest lower-numbered aligned slice
    best_match = None
    best_diff = float("inf")
    
    for s in aligned_slices:
        num = extract_slice_number(s)
        if num > 0 and num < current_num:
            diff = current_num - num
            if diff < best_diff:
                best_diff = diff
                best_match = s
    
    if best_match:
        return g.get_transform(best_match, invivo_node)
    return None


print("Alignment utilities loaded")
print("  - get_initial_transform_3d_base()")
print("  - get_initial_transform_3d_movable()")
print("  - save_alignment(slice_name, transform)")
print("  - get_previous_transform(slice_name)")

---

## 6. Alignment Mode A: Padded, 3D as Base (with crop)

**Use when:** You need point-based transforms (T, R, N, V, P) on rice-paper slices.

**How it works:** 
- Pads the 1-voxel slice to give it z-extent, enabling point-based fitting
- `crop=True` limits rendering to in-vivo bounds (prevents 7+ GB RAM explosion)

**RAM usage:** ~400-500 MB (vs 7+ GB without crop)

**Transform type:** TranslateRotate (point-based)

**Controls:**
- Click corresponding points on base and movable
- Press 'q' to quit and save

In [ ]:
# =============================================================================
# MODE A: PADDED, 3D AS BASE
# =============================================================================

# Configuration
PAD_Z = 25  # Slices above and below

# Get selected slice
slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode: Padded, 3D as base")
print()

# Load images
slice_img = g.get_image(slice_name)
invivo_img = g.get_image(invivo_node)

print(f"Slice shape: {slice_img.shape}")
print(f"In-vivo shape: {invivo_img.shape}")

# Create padded array
padding_value = slice_img.mean() * 0.90
padded = np.full((2*PAD_Z + 1, *slice_img.shape[1:]), padding_value, dtype=slice_img.dtype)
padded[PAD_Z] = slice_img[0]  # Original slice in middle

print(f"\nPadded shape: {padded.shape}")
print(f"Memory: {padded.nbytes / 1e6:.1f} MB")

# Get initial transform (or carry forward from previous slice)
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform
    print(f"\n✓ Using transform from previous aligned slice")
else:
    initial_transform = get_initial_transform_3d_base()
    print(f"\n✓ Using default initial transform")

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print("Movable: padded 2D slice (green)")
print("Base: 3D in-vivo stack (magenta)")
print("Press 'q' to quit when done")
print()

transform = ca_gui.align_interactive(
    nodes_movable=padded,
    nodes_fixed=invivo_img,
    transform=initial_transform,
    graph=None
)

# Auto-save
save_alignment(slice_name, transform)

# Cleanup
del padded
import gc
gc.collect()

print("\n✓ Alignment complete")

---

## 7. Alignment Mode B: No Padding, 3D as Base (Parametric, with crop)

**Use when:** You only need slider-based alignment, no keypoints.

**How it works:** 
- Uses the raw 1-voxel slice directly (no padding)
- `crop=True` limits rendering to in-vivo bounds
- Parametric (slider-based) transforms only

**RAM usage:** Minimal (~50 MB for slice + in-vivo)

**Transform type:** TranslateRotateFixed (slider-based)

**Controls:**
- Use sliders to adjust rotation and translation
- Press 'q' to quit and save

**Note:** No point clicking in this mode. For point-based, use Mode A or C.

In [ ]:
# =============================================================================
# MODE B: NO PADDING, 3D AS BASE (PARAMETRIC ONLY)
# =============================================================================

# Get selected slice
slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode: No padding, 3D as base")
print()

# Load images (no padding)
slice_img = g.get_image(slice_name)
invivo_img = g.get_image(invivo_node)

print(f"Slice shape: {slice_img.shape}")
print(f"In-vivo shape: {invivo_img.shape}")
print(f"Memory: {slice_img.nbytes / 1e6:.1f} MB (no padding)")

# Get initial transform
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform
    print(f"\n✓ Using transform from previous aligned slice")
else:
    initial_transform = get_initial_transform_3d_base()
    print(f"\n✓ Using default initial transform")

# Warning
print("\n" + "!"*60)
print("NOTE: No padding - point-based transforms (T, R, P) may not work well")
print("Use parametric transforms (t, r) or switch to Mode A for point-based")
print("!"*60)

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print("Movable: 2D slice (green)")
print("Base: 3D in-vivo stack (magenta)")
print("Press 'q' to quit when done")
print()

transform = ca_gui.align_interactive(
    nodes_movable=slice_img,
    nodes_fixed=invivo_img,
    transform=initial_transform,
    graph=None
)

# Auto-save
save_alignment(slice_name, transform)

print("\n✓ Alignment complete")

---

## 8. Alignment Mode C: Padded 2D as Base (3D Movable)

**Use when:** You want to scroll through the 3D volume to find where the 2D slice fits.

**How it works:** 
- 2D slice is base with padding (PAD_Z=25, same as Mode A)
- 3D volume is movable
- No pre-transforms - manually rotate in napari as needed
- Graph stores both directions automatically

**RAM usage:** ~300-400 MB (in-vivo ~200MB + padded slice ~100MB)

**Benefits:**
- All transforms work (T, R, N, V, P) due to padding
- Can scroll through 3D to find matching z-location
- Full manual control over orientation

In [ ]:
# =============================================================================
# MODE C: PADDED 2D AS BASE (3D MOVABLE)
# =============================================================================

# Configuration
PAD_Z = 25  # Slices above and below (same as Mode A)

# Get selected slice
slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode: Padded 2D as base (3D MOVABLE)")
print()

# Load images (note: roles are swapped!)
slice_img = g.get_image(slice_name)
invivo_img = g.get_image(invivo_node)

print(f"Base (2D slice): {slice_img.shape}")
print(f"Movable (3D volume): {invivo_img.shape}")

# =============================================================================
# PADDING FOR 2D BASE (same approach as Mode A)
# =============================================================================
# Pad the base slice to give it z-extent, enabling point-based fitting.
# The actual slice sits at z = PAD_Z in the padded volume.
# =============================================================================
padding_value = slice_img.mean() * 0.90
padded_base = np.full((2*PAD_Z + 1, *slice_img.shape[1:]), padding_value, dtype=slice_img.dtype)
padded_base[PAD_Z] = slice_img[0]  # Original slice in middle

print(f"Padded base shape: {padded_base.shape}")
print(f"  Actual slice at z = {PAD_Z}")
print(f"Memory: {padded_base.nbytes / 1e6:.1f} MB")

# Initial transform for movable (3D volume) - start with identity
initial_transform = ca.Identity()

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print("Movable: 3D in-vivo volume (green)")
print("Base: 2D slice (magenta) - padded")
print("Use napari controls to manually rotate/adjust view")
print("Press 'q' to quit when done")
print()

transform_inv = ca_gui.align_interactive(
    nodes_movable=invivo_img,   # 3D as movable
    nodes_fixed=padded_base,    # 2D as base (padded)
    transform=initial_transform,
    graph=None
)

# Save transform
try:
    g.add_edge(invivo_node, slice_name, transform_inv, update=True)
except:
    g.add_edge(invivo_node, slice_name, transform_inv)

g.save()

# Update tracking
if slice_name not in aligned_slices:
    aligned_slices.append(slice_name)
if slice_name in unaligned_slices:
    unaligned_slices.remove(slice_name)
slice_dropdown.options = get_slice_options()

print(f"\n✓ Saved: {invivo_node} ↔ {slice_name} (bidirectional)")
print(f"  Graph saved to: {GRAPH_PATH.name}")

# Cleanup
del padded_base
import gc
gc.collect()

print("\n✓ Alignment complete")

---

## 9. Nonlinear Refinement: Triangulation2D (with crop)

**Use after:** Initial affine alignment (Mode A, B, or C)

**What it does:** Adds nonlinear warping to correct edge distortions.

**How it works:**
- Uses padding (required for point-based Triangulation2D)
- `crop=True` limits rendering to in-vivo bounds
- Starts from existing transform, adds nonlinear refinement

**How to use:**
1. Select an already-aligned slice
2. Run this cell
3. Click corresponding points on edges that need warping
4. Press 'q' to quit and save

In [ ]:
# =============================================================================
# NONLINEAR REFINEMENT: TRIANGULATION (requires padding)
# =============================================================================

# Get selected slice
slice_name = get_selected_slice()

# Check if slice is aligned
if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
    print("Run one of the alignment modes first.")
else:
    print(f"Refining: {slice_name}")
    print(f"Mode: Nonlinear refinement (N or V transforms)")
    print()
    
    # Get existing transform
    existing_transform = g.get_transform(slice_name, invivo_node)
    print(f"Existing transform: {type(existing_transform).__name__}")
    
    # Load images
    slice_img = g.get_image(slice_name)
    invivo_img = g.get_image(invivo_node)
    
    # For point-based nonlinear transforms, we need padding
    PAD_Z = 25
    padding_value = slice_img.mean() * 0.90
    padded = np.full((2*PAD_Z + 1, *slice_img.shape[1:]), padding_value, dtype=slice_img.dtype)
    padded[PAD_Z] = slice_img[0]
    
    print(f"\nPadded shape: {padded.shape}")
    
    # Launch GUI with existing transform
    # Use N (projected triangulation) or V (3D triangulation) from the menu
    print("\n" + "="*60)
    print("LAUNCHING GUI FOR REFINEMENT")
    print("="*60)
    print("Starting from existing transform")
    print("Use 'N' for projected triangulation or 'V' for 3D triangulation")
    print("Press 'q' to quit when done")
    print()
    
    refined_transform = ca_gui.align_interactive(
        nodes_movable=padded,
        nodes_fixed=invivo_img,
        transform=existing_transform,
        graph=None
    )
    
    # Auto-save
    save_alignment(slice_name, refined_transform)
    
    # Cleanup
    del padded
    import gc
    gc.collect()
    
    print("\n✓ Refinement complete")

---

## 10. GraphViewer: See All Aligned Slices

Visualize all aligned slices together in the in-vivo coordinate space.

In [ ]:
# =============================================================================
# GRAPHVIEWER: VISUALIZE ALL ALIGNED SLICES
# =============================================================================

print("="*60)
print("GRAPHVIEWER")
print("="*60)
print(f"Aligned slices: {len(aligned_slices)}")
print()

if len(aligned_slices) == 0:
    print("No slices aligned yet. Run an alignment mode first.")
else:
    if GraphViewer is None:
        # Fallback if GraphViewer not available
        print("GraphViewer not available in this version.")
        print("Using manual napari viewer instead...")
        print()
        
        import napari
        
        v = napari.Viewer()
        
        # Add in-vivo reference
        invivo_img = g.get_image(invivo_node)
        v.add_image(invivo_img, name=invivo_node, colormap="gray", opacity=0.7)
        
        # Add each aligned slice, transformed to invivo space
        colormaps = ["red", "green", "blue", "cyan", "magenta", "yellow"]
        
        for i, slice_name in enumerate(sorted(aligned_slices)):
            try:
                # Get slice and transform
                slice_img = g.get_image(slice_name)
                transform = g.get_transform(slice_name, invivo_node)
                
                # Apply transform
                transformed = transform.transform_image(slice_img)
                
                # Get origin if available
                origin = transformed.origin if hasattr(transformed, "origin") else [0, 0, 0]
                
                # Add to viewer
                cmap = colormaps[i % len(colormaps)]
                v.add_image(
                    transformed, 
                    name=slice_name, 
                    colormap=cmap, 
                    opacity=0.5,
                    translate=origin
                )
                print(f"  Added: {slice_name} ({cmap})")
            except Exception as e:
                print(f"  Failed: {slice_name} - {e}")
        
        print("\n✓ Viewer opened")
        napari.run()
        
    else:
        # Use GraphViewer
        print("Opening GraphViewer in invivo_ref space...")
        print()
        
        v = GraphViewer(g, space=invivo_node)
        
        # Add in-vivo reference
        v.add_image(invivo_node, colormap="gray", opacity=0.7)
        
        # Add aligned slices
        colormaps = ["red", "green", "blue", "cyan", "magenta", "yellow"]
        
        for i, slice_name in enumerate(sorted(aligned_slices)):
            try:
                cmap = colormaps[i % len(colormaps)]
                v.add_image(slice_name, colormap=cmap, opacity=0.5)
                print(f"  Added: {slice_name} ({cmap})")
            except Exception as e:
                print(f"  Failed: {slice_name} - {e}")
        
        print("\n✓ GraphViewer opened")

---

## 11. Alignment Status Summary

Quick overview of all slices and their alignment status.

In [ ]:
# =============================================================================
# ALIGNMENT STATUS SUMMARY
# =============================================================================

# Refresh alignment status from graph
aligned_slices = []
unaligned_slices = []

for s in slice_nodes:
    if s in g.edges and invivo_node in g.edges[s]:
        aligned_slices.append(s)
    else:
        unaligned_slices.append(s)

print("="*60)
print("ALIGNMENT STATUS")
print("="*60)
print(f"Total slices: {len(slice_nodes)}")
print(f"Aligned: {len(aligned_slices)} ({100*len(aligned_slices)/len(slice_nodes):.1f}%)")
print(f"Remaining: {len(unaligned_slices)}")
print()

# Progress bar
progress = len(aligned_slices) / len(slice_nodes)
bar_width = 40
filled = int(bar_width * progress)
bar = "█" * filled + "░" * (bar_width - filled)
print(f"Progress: [{bar}] {100*progress:.1f}%")
print()

# List aligned slices
if aligned_slices:
    print("Aligned slices:")
    for s in sorted(aligned_slices):
        t = g.get_transform(s, invivo_node)
        print(f"  ✓ {s}: {type(t).__name__}")
    print()

# List next few unaligned
if unaligned_slices:
    print("Next unaligned slices:")
    for s in sorted(unaligned_slices)[:5]:
        print(f"  ○ {s}")
    if len(unaligned_slices) > 5:
        print(f"  ... and {len(unaligned_slices) - 5} more")

print("="*60)

---

## 12. Graph Structure Visualization

View the graph structure as a node-edge diagram.

In [ ]:
# =============================================================================
# GRAPH STRUCTURE VISUALIZATION
# =============================================================================

print(f"Graph: {g.name}")
print(f"Nodes: {len(g.nodes)}")
print(f"Edges: {sum(len(e) for e in g.edges.values()) // 2}")
print()

# Visualize (uses SVG via patch)
g.visualise()